# Production ML Pipeline: Customer Churn Prediction
This notebook demonstrates a complete, production-grade Machine Learning pipeline implementing best practices for data separation, column transformations, cross-validation, and out-of-sample evaluation.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
# 1. Ingest Customer Data
np.random.seed(42)
n_samples = 1200

df = pd.DataFrame({
    'customer_id': [f'CUST_{i:04d}' for i in range(n_samples)],
    'tenure_months': np.random.randint(1, 60, n_samples),
    'monthly_charges': np.random.uniform(25.0, 110.0, n_samples),
    'contract_type': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples),
    'tech_support': np.random.choice(['Yes', 'No'], n_samples),
    'churn': np.random.choice([0, 1], n_samples, p=[0.75, 0.25])
})

df.head()

In [3]:
# 2. Feature and Target Separation
X = df.drop(columns=['churn', 'customer_id'])
y = df['churn']

numeric_features = ['tenure_months', 'monthly_charges']
categorical_features = ['contract_type', 'tech_support']

# Strict Train / Test Split before fitting any transformers
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)
print(f'Training shape: {X_train.shape}, Test shape: {X_test.shape}')

In [4]:
# 3. Encapsulated Preprocessing & Estimator Pipeline
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

In [5]:
# 4. Model Training & Evaluation on Unseen Test Partition
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print('Test ROC-AUC Score:', roc_auc_score(y_test, y_prob))
print('\nClassification Report:\n', classification_report(y_test, y_pred))